In [1]:

# 1. Verificar ambiente e Instalar Dependências
import os
import sys

try:
    from google.colab import drive
    IN_COLAB = True
    print("Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally")

if IN_COLAB:
    if not os.path.exists('/content/ufc-easytpp'):
        !git clone https://github.com/hugoramos/ufc-easytpp.git /content/ufc-easytpp
    
    !pip install -q torch numpy matplotlib pandas datasets pyyaml transformers

    %cd /content/ufc-easytpp
    if '/content/ufc-easytpp' not in sys.path:
        sys.path.append('/content/ufc-easytpp')

else:
    local_path = '/Users/hugoramossoares/Sites/EasyTemporalPointProcess'
    if os.path.exists(local_path):
        os.chdir(local_path)
        if local_path not in sys.path:
            sys.path.append(local_path)

import torch
import numpy as np
import matplotlib.pyplot as plt
from easy_tpp.model.torch_model.torch_nhp import NHP
from easy_tpp.model.torch_model.torch_thp import THP
print("Bibliotecas carregadas.")


Running locally
Bibliotecas carregadas.


In [ ]:

# 2. Gerar Dataset Sintético com Dependência de Longo Alcance (Power Law Kernel)
#
# Processo: Hawkes com Kernel de Lei de Potência (Power Law)
# Função de Intensidade: lambda(t) = mu + sum( alpha * (delta + t - t_i)^(-beta) )
#
# Este kernel decai muito mais lentamente que o exponencial, criando dependências
# de longo prazo que são tipicamente difíceis para RNNs (NHP) capturarem,
# mas teoricamente acessíveis para Transformers (THP) via atenção.

def power_law_kernel(t, t_history, alpha, beta, delta):
    if len(t_history) == 0:
        return 0
    # (delta + t - t_i)^(-beta)
    dt = t - t_history
    val = alpha * np.power(delta + dt, -beta)
    return np.sum(val)

def simulate_power_law_hawkes(mu, alpha, beta, delta, max_time=100.0):
    events = []
    t = 0
    
    # Upper bound inicial grosseiro
    # lambda(t) é decrescente entre eventos.
    # Logo após um evento, é o pico.
    
    history = np.array([])
    
    while t < max_time:
        # Calcular lambda atual (pico imediato para upper bound)
        # Se t=0, lambda = mu
        # Se t > 0, lambda(t) <= lambda(last_t + epsilon)
        
        # Ogata's Thinning
        # 1. Upper bound lambda_bar
        # Usamos o valor atual da intensidade como upper bound para o próximo intervalo
        # (pois o kernel é monotônico decrescente)
        
        current_intensity = mu + power_law_kernel(t, history, alpha, beta, delta)
        lambda_bar = current_intensity
        
        # 2. Gerar tempo candidato
        u = np.random.uniform(0, 1)
        w = -np.log(u) / lambda_bar
        t += w
        
        if t >= max_time:
            break
            
        # 3. Rejeição
        # Calcular intensidade REAL em t
        true_intensity = mu + power_law_kernel(t, history, alpha, beta, delta)
        
        d = np.random.uniform(0, 1)
        if d * lambda_bar <= true_intensity:
            # Aceita
            events.append(t)
            history = np.array(events)
            
    return events

def generate_long_memory_dataset(num_seqs, max_time=200):
    print(f"Gerando {num_seqs} sequências com Power Law Kernel...")
    dataset = []
    
    # Parâmetros que favorecem memória longa
    mu = 0.5
    alpha = 0.8  # Alta excitação
    beta = 1.5   # Decay lento (beta > 1 para estabilidade, mas próximo de 1 é cauda longa)
    delta = 1.0  # Shift para evitar singularidade em 0
    
    for _ in range(num_seqs):
        times = simulate_power_law_hawkes(mu, alpha, beta, delta, max_time=max_time)
        
        if len(times) < 10: continue # Ignorar sequências muito curtas
        
        # Formato EasyTPP (List of Dicts)
        # Single event type (univariate) para focar na dinâmica temporal
        times = np.array(times)
        
        # Criar time_delta
        time_since_last = np.concatenate([[times[0]], np.diff(times)])
        
        # Type event (tudo 0)
        types = np.zeros_like(times, dtype=int)
        
        dataset.append({
            'time_since_start': times,
            'time_since_last_event': time_since_last,
            'type_event': types
        })
        
    print(f"Geradas {len(dataset)} sequências válidas.")
    lens = [len(x['time_since_start']) for x in dataset]
    print(f"Comp Médio: {np.mean(lens):.1f} eventos")
    return dataset

# Gerar dados
MAX_TIME = 200 # Tempo total da simulação
data = generate_long_memory_dataset(1000, max_time=MAX_TIME) # 1000 seqs

# Split 80/20
split_idx = int(0.8 * len(data))
train_data = data[:split_idx]
test_data = data[split_idx:]

print(f"Treino: {len(train_data)}")
print(f"Teste:  {len(test_data)}")

# Calcular escala para normalização
all_deltas = [d for seq in train_data for d in seq['time_since_last_event']]
TIME_SCALE = np.mean(all_deltas)
print(f"Time Scale (Mean Delta): {TIME_SCALE:.4f}")

NUM_TYPES = 1 # Univariado


Gerando 1000 sequências com Power Law Kernel...


In [ ]:

# 3. Configuração
def collate_fn_factory(time_scale):
    def collate_fn(batch_list):
        time_seqs = []
        time_delta_seqs = []
        type_seqs = []
        max_len = 0
        for item in batch_list:
            ts = item['time_since_start']
            td = item['time_since_last_event']
            ev = item['type_event']
            if len(ts) > max_len: max_len = len(ts)
            
            # Normalize
            ts_norm = (torch.tensor(ts, dtype=torch.float64) - ts[0]) / time_scale
            td_norm = torch.tensor(td, dtype=torch.float64) / time_scale
            
            time_seqs.append(ts_norm.float())
            time_delta_seqs.append(td_norm.float())
            type_seqs.append(torch.tensor(ev, dtype=torch.long))
    
        batch_size = len(batch_list)
        pad_time = torch.zeros(batch_size, max_len)
        pad_delta = torch.zeros(batch_size, max_len)
        pad_type = torch.zeros(batch_size, max_len, dtype=torch.long)
        attention_mask = torch.zeros(batch_size, max_len, max_len)
        batch_non_pad_mask = torch.zeros(batch_size, max_len)
        
        for i in range(batch_size):
            l = len(time_seqs[i])
            pad_time[i, :l] = time_seqs[i]
            pad_delta[i, :l] = time_delta_seqs[i]
            pad_type[i, :l] = type_seqs[i]
            batch_non_pad_mask[i, :l] = 1
            
            # Mask
            causal_mask = torch.triu(torch.ones(max_len, max_len), diagonal=1)
            causal_mask[:, l:] = 1
            causal_mask[l:, :] = 1
            attention_mask[i] = causal_mask
    
        return (pad_time, pad_delta, pad_type, batch_non_pad_mask, attention_mask)
    return collate_fn

class ModelConfig:
    def __init__(self, num_types, hidden_size=64):
        self.num_event_types = num_types
        self.num_event_types_pad = num_types + 1
        self.pad_token_id = num_types
        self.hidden_size = hidden_size
        self.time_emb_size = hidden_size
        self.num_layers = 2
        self.num_heads = 4
        self.dropout_rate = 0.1
        self.use_ln = True
        self.gpu = 0 if torch.cuda.is_available() else -1
        self.thinning = type('Thinning',(),{'num_sample':100,'num_exp':500,'over_sample_rate':10.0,'patience_counter':5,'num_samples_boundary':20,'dtime_max':5.0})()
        self.model_specs = {'beta': 1.0, 'bias': True}


In [ ]:

# 4. Treinamento
import time

def collate_fn_factory(time_scale):
    def collate_fn(batch_list):
        time_seqs = []
        time_delta_seqs = []
        type_seqs = []
        max_len = 0
        for item in batch_list:
            ts = item['time_since_start']
            td = item['time_since_last_event']
            ev = item['type_event']
            if len(ts) > max_len: max_len = len(ts)
            
            # Normalize
            ts_norm = (torch.tensor(ts, dtype=torch.float64) - ts[0]) / time_scale
            td_norm = torch.tensor(td, dtype=torch.float64) / time_scale
            
            time_seqs.append(ts_norm.float())
            time_delta_seqs.append(td_norm.float())
            type_seqs.append(torch.tensor(ev, dtype=torch.long))
    
        batch_size = len(batch_list)
        pad_time = torch.zeros(batch_size, max_len)
        pad_delta = torch.zeros(batch_size, max_len)
        pad_type = torch.zeros(batch_size, max_len, dtype=torch.long)
        attention_mask = torch.zeros(batch_size, max_len, max_len)
        batch_non_pad_mask = torch.zeros(batch_size, max_len)
        
        for i in range(batch_size):
            l = len(time_seqs[i])
            pad_time[i, :l] = time_seqs[i]
            pad_delta[i, :l] = time_delta_seqs[i]
            pad_type[i, :l] = type_seqs[i]
            batch_non_pad_mask[i, :l] = 1
            
            # Mask
            causal_mask = torch.triu(torch.ones(max_len, max_len), diagonal=1)
            causal_mask[:, l:] = 1
            causal_mask[l:, :] = 1
            attention_mask[i] = causal_mask
    
        return (pad_time, pad_delta, pad_type, batch_non_pad_mask, attention_mask)
    return collate_fn

class ModelConfig:
    def __init__(self, num_types, hidden_size=64):
        self.num_event_types = num_types
        self.num_event_types_pad = num_types + 1
        self.pad_token_id = num_types
        self.hidden_size = hidden_size
        self.time_emb_size = hidden_size
        self.num_layers = 2
        self.num_heads = 4
        self.dropout_rate = 0.1
        self.use_ln = True
        self.gpu = 0 if torch.cuda.is_available() else -1
        self.thinning = type('Thinning',(),{'num_sample':100,'num_exp':500,'over_sample_rate':10.0,'patience_counter':5,'num_samples_boundary':20,'dtime_max':5.0})()
        self.model_specs = {'beta': 1.0, 'bias': True}

# Add metrics computation function
def compute_metrics(model, test_ds, collate_fn):
    model.eval()
    total_rmse = 0
    total_events = 0
    subset = test_ds[:200]
    
    with torch.no_grad():
        for i in range(0, len(subset), 32):
            batch = collate_fn(subset[i:i+32])
            _, time_delta_target, _, mask_target, _ = batch
            
            # predict_one_step_at_every_event returns (dtimes, types)
            dtimes_pred, _ = model.predict_one_step_at_every_event(batch)
            
            # Align: predictions align with targets[1:]
            # time_delta_target shape: [batch, len]
            target = time_delta_target[:, 1:]
            mask = mask_target[:, 1:]
            
            se = ((dtimes_pred - target) ** 2) * mask
            total_rmse += se.sum().item()
            total_events += mask.sum().item()
            
    return np.sqrt(total_rmse / (total_events+1e-9))

def train_loop(model_class, name, config, train_ds, test_ds, time_scale, epochs=15):
    print(f"\n>>> Treinando {name}...")
    model = model_class(config)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    collate = collate_fn_factory(time_scale)
    
    nll_hist = []
    
    for ep in range(epochs):
        model.train()
        total_loss = 0
        total_ev = 0
        
        # Full epoch with shuffle
        indices = np.random.permutation(len(train_ds))
        for i in range(0, len(indices), 64):
            batch_list = [train_ds[k] for k in indices[i:i+64]]
            batch = collate(batch_list)
            
            opt.zero_grad()
            loss, n = model.loglike_loss(batch)
            
            if torch.isnan(loss): 
                # print("NaN Loss")
                continue
                
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            
            total_loss += loss.item()
            total_ev += n
            
        train_nll = total_loss / (total_ev + 1e-9)
        
        # Val
        model.eval()
        val_loss = 0
        val_ev = 0
        with torch.no_grad():
            batch = collate(test_ds[:100])
            l, n = model.loglike_loss(batch)
            val_loss += l.item()
            val_ev += n
        val_nll = val_loss / (val_ev + 1e-9)
        
        nll_hist.append(val_nll)
        print(f"  Ep {ep+1} | Train NLL: {train_nll:.4f} | Val NLL: {val_nll:.4f}")
        
    rmse = compute_metrics(model, test_ds, collate)
    print(f"  >>> Final RMSE: {rmse:.4f}")
    return nll_hist, rmse, model

config = ModelConfig(NUM_TYPES, hidden_size=128) # Maior capacidade

# Treinar NHP
hist_nhp, rmse_nhp, model_nhp = train_loop(NHP, "NHP (RNN)", config, train_data, test_data, TIME_SCALE, epochs=15)

# Treinar THP
hist_thp, rmse_thp, model_thp = train_loop(THP, "THP (Transformer)", config, train_data, test_data, TIME_SCALE, epochs=15)

# Plot
plt.figure(figsize=(10, 5))
plt.plot(hist_nhp, label=f'NHP (RMSE={rmse_nhp:.4f})')
plt.plot(hist_thp, label=f'THP (RMSE={rmse_thp:.4f})')
plt.title('Comparação NHP vs THP - Dataset Power Law (Long Memory)')
plt.ylabel('NLL (Validation)')
plt.xlabel('Epoch')
plt.legend()
plt.show()


In [ ]:

# 5. Visualizar Intensidade Prevista vs Real
# Vamos ver se os modelos capturam o decaimento lento (Power Law)

def true_power_law_intensity(t, history, mu=0.5, alpha=0.8, beta=1.5, delta=1.0):
    val = mu + np.sum(alpha * np.power(delta + t - history, -beta))
    return val

seq = test_data[0]
ts = seq['time_since_start']
interval_idx = 10
t_start = ts[interval_idx]
t_end = ts[interval_idx+1]

t_grid = np.linspace(t_start, t_end, 100)
lambda_true = []
history = ts[:interval_idx+1]

for t in t_grid:
    lambda_true.append(true_power_law_intensity(t, history))

# Predictions
# (Simplificado para usar a função interna dos modelos se possível, 
# mas aqui vamos plotar apenas o True para ilustrar o desafio)
plt.figure(figsize=(8, 4))
plt.plot(t_grid, lambda_true, 'k--', label='True Power Law')
plt.title(f'Intensidade Verdadeira no Intervalo {interval_idx}')
plt.xlabel('Tempo')
plt.ylabel('Intensidade')
plt.legend()
plt.show()

print("Para ver as predições dos modelos, use a função plot_intensity_comparison definida no notebook anterior.")
